# 🛠️ CodeFix-Env — Quick Start
**A PyTorch-based RL environment for automated code debugging.**

Built for the Meta PyTorch × OpenEnv Hackathon.

[![GitHub](https://img.shields.io/badge/GitHub-codefix--env-black?logo=github)](https://github.com/dhakarshailendra829/codefix-env)

This notebook covers:
1. Installation
2. Direct environment usage (no server)
3. Exploring tasks
4. Running a full debugging episode
5. Using the HTTP client (with server)

## 1. Installation

In [ ]:
# Install codefix-env from GitHub
!pip install git+https://github.com/dhakarshailendra829/codefix-env.git

# Or for local development (run from repo root):
# !pip install -e .[dev]

In [ ]:
import codefix_env
print(f'CodeFix-Env version: {codefix_env.__version__}')
from codefix_env import task_count
print(f'Available tasks: {task_count()}')

## 2. Explore Available Tasks

In [ ]:
from codefix_env import list_tasks, load_task, Difficulty

# Show all tasks
print(f"{'ID':<45} {'Difficulty':<10} {'Bug Category'}")
print('-' * 80)
for task in list_tasks():
    print(f"{task.id:<45} {task.difficulty:<10} {task.bug_category}")

In [ ]:
# Inspect a specific task
task = load_task('medium-002-binary-search')
print(f'Title: {task.title}')
print(f'Description: {task.description}')
print(f'Difficulty: {task.difficulty}')
print(f'Bug Category: {task.bug_category}')
print(f'Test cases: {task.num_tests}')
print(f'Max steps: {task.max_steps}')
print()
print('=== BUGGY CODE ===')
print(task.buggy_code)
print('=== HINTS ===')
for i, hint in enumerate(task.hints, 1):
    print(f'Hint {i}: {hint}')

## 3. Direct Environment Usage
No server needed. Use `CodeFixEnvironment` directly in Python.

In [ ]:
from codefix_env import CodeFixEnvironment, CodeFixAction, ActionType

# Create environment
env = CodeFixEnvironment()

# Reset with a specific task
obs = env.reset(task_id='easy-001-missing-return')

print(f'Task: {obs.task_id}')
print(f'Steps remaining: {obs.steps_remaining}')
print(f'Tests: {obs.tests_passed}/{obs.tests_total}')
print()
print('Buggy code:')
for i, line in enumerate(obs.current_code.splitlines(), 1):
    print(f'  {i:2d} | {line}')

In [ ]:
# Step 1: Run tests to see what's failing
result = env.step(CodeFixAction(action_type=ActionType.RUN_TESTS))
obs = result.observation

print(f'Tests passing: {obs.tests_passed}/{obs.tests_total}')
print(f'Reward: {result.reward:.3f}')
print(f'Test output:')
print(obs.test_output)

In [ ]:
# Step 2: Get a hint
result = env.step(CodeFixAction(action_type=ActionType.GET_HINT))
print(result.observation.feedback)
print(f'Reward: {result.reward:.3f} (hint penalty)')

In [ ]:
# Step 3: Fix the bug
# easy-001: function never returns result - insert return statement after line 3
result = env.step(CodeFixAction(
    action_type=ActionType.INSERT_LINE,
    line_number=3,
    new_content='    return result',
    reasoning='The function computes result but never returns it.',
))
print(f'Reward after fix: {result.reward:.3f}')
print()
print('Current code:')
for i, line in enumerate(result.observation.current_code.splitlines(), 1):
    print(f'  {i:2d} | {line}')

In [ ]:
# Step 4: Run tests again
result = env.step(CodeFixAction(action_type=ActionType.RUN_TESTS))
obs = result.observation
print(f'Tests passing: {obs.tests_passed}/{obs.tests_total}')
print(f'All pass: {obs.all_tests_pass}')
print(obs.test_output)

In [ ]:
# Step 5: Submit the fix
result = env.step(CodeFixAction(action_type=ActionType.SUBMIT_FIX))
print(f'Final score: {result.reward:.3f}')
print(f'Solved: {result.observation.all_tests_pass}')
print(f'Episode done: {result.done}')
print(f'Total steps used: {result.observation.step_count}')

## 4. Full Episode Loop
This is the standard RL training loop pattern.

In [ ]:
from codefix_env import CodeFixEnvironment, CodeFixAction, ActionType, Difficulty

def run_episode(task_id=None, difficulty=None, verbose=True):
    """Run one episode with a simple greedy agent."""
    env = CodeFixEnvironment()
    obs = env.reset(task_id=task_id, difficulty=difficulty)

    if verbose:
        print(f'Task: {obs.task_id} | Difficulty: {obs.difficulty}')
        print(f'Bug: {obs.bug_category}')

    total_reward = 0.0
    done = False
    step = 0

    while not done:
        step += 1
        # Simple agent: run_tests first, then submit
        if step == 1:
            action = CodeFixAction(action_type=ActionType.RUN_TESTS)
        else:
            action = CodeFixAction(action_type=ActionType.SUBMIT_FIX)

        result = env.step(action)
        total_reward += result.reward
        done = result.done or result.truncated

        if verbose:
            print(f'  Step {step}: {action.action_type} → reward={result.reward:.3f} | ""
                  f'tests={result.observation.tests_passed}/{result.observation.tests_total}')

    print(f'\nFinal: solved={result.observation.all_tests_pass} | ""
          f'score={result.observation.score_so_far:.3f} | total_reward={total_reward:.3f}')
    return result

# Run an easy episode
run_episode(task_id='easy-008-inverted-bool')

## 5. HTTP Client (Requires Running Server)

First start the server in a terminal:
```bash
# Windows
set PYTHONPATH=src
uvicorn server.app:app --reload --port 8000

# Linux/Mac
PYTHONPATH=src uvicorn server.app:app --reload --port 8000
```

In [ ]:
import asyncio
from codefix_env import CodeFixClient, CodeFixAction, ActionType

async def http_demo():
    async with CodeFixClient('http://localhost:8000') as env:
        # Check health
        health = await env.health()
        print(f'Server: {health}')

        # Reset
        obs = await env.reset(task_id='easy-001-missing-return')
        print(f'Task: {obs.task_id}')

        # Step
        result = await env.step(CodeFixAction(action_type=ActionType.RUN_TESTS))
        print(f'Tests: {result.observation.tests_passed}/{result.observation.tests_total}')

# Uncomment when server is running:
# await http_demo()  # in Jupyter/Colab
# asyncio.run(http_demo())  # in plain Python

print('Start the server first, then uncomment the line above.')

## 6. Quick Benchmark

In [ ]:
from codefix_env import CodeFixEnvironment, CodeFixAction, ActionType, list_tasks

# Quick benchmark: run_tests then submit on all easy tasks
env = CodeFixEnvironment()
results = []

for task in list_tasks(difficulty='easy'):
    obs = env.reset(task_id=task.id)
    env.step(CodeFixAction(action_type=ActionType.RUN_TESTS))
    result = env.step(CodeFixAction(action_type=ActionType.SUBMIT_FIX))
    results.append({
        'task': task.id,
        'solved': result.observation.all_tests_pass,
        'score': round(result.observation.score_so_far, 3),
    })

print(f"{'Task':<45} {'Solved':<8} Score")
print('-' * 65)
for r in results:
    icon = '✅' if r['solved'] else '❌'
    print(f"{r['task']:<45} {icon:<8} {r['score']}")

print()
solved = sum(r['solved'] for r in results)
print(f'Solved: {solved}/{len(results)} ({100*solved/len(results):.0f}%)')
print(f'Avg score: {sum(r["score"] for r in results)/len(results):.3f}')